# 🥞 Stacking — Meta-Learning Ensemble

> **Folder:** `08_Ensemble_Learning`  
> **Notebook:** `stacking.ipynb`  
> **Author:** Hamna Munir

---

## 🎯 Objectives

By the end of this notebook, you will:

- Understand **how stacking differs from bagging and boosting**
- Build a **StackingClassifier** with diverse base learners
- Use **cross-val predictions** (out-of-fold) to train the meta-learner
- Compare **different meta-learners**: Logistic Regression, Ridge, RF
- Implement **manual stacking** from scratch for full transparency
- Build **multi-level stacking** (Level 0 -> Level 1 -> Level 2)
- Apply stacking to **regression** with StackingRegressor
- Compare stacking vs individual models and voting ensembles

---

## 📚 Techniques Covered

| # | Technique | Key Insight |
|---|-----------|-------------|
| 1 | Dataset Setup | Classification + Regression |
| 2 | Stacking Intuition | Out-of-fold predictions prevent leakage |
| 3 | StackingClassifier | sklearn API — base + meta learner |
| 4 | Meta-Learner Comparison | LR vs Ridge vs RF as meta-learner |
| 5 | Manual Stacking | Step-by-step OOF implementation |
| 6 | Base Learner Diversity | Why diverse models stack better |
| 7 | Passthrough Features | Include original features in meta-learner |
| 8 | Multi-Level Stacking | Level 0 -> Level 1 -> Level 2 |
| 9 | StackingRegressor | Regression stacking |
| 10 | Voting vs Stacking | Learned vs fixed combination |
| 11 | Full Leaderboard | All ensembles vs single models |
| 12 | Summary & Golden Rules | Key takeaways |


---
## ⚙️ 0. Setup & Imports

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.datasets import make_classification, make_regression
from sklearn.model_selection import (
    train_test_split, StratifiedKFold, KFold,
    cross_val_score, cross_val_predict,
)
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.ensemble import (
    StackingClassifier, StackingRegressor,
    VotingClassifier, VotingRegressor,
    RandomForestClassifier, GradientBoostingClassifier,
    BaggingClassifier,
)
from sklearn.linear_model import (
    LogisticRegression, RidgeCV, Ridge, Lasso,
)
from sklearn.svm import SVC, SVR
from sklearn.neighbors import KNeighborsClassifier, KNeighborsRegressor
from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import (
    accuracy_score, f1_score, roc_auc_score,
    r2_score, mean_squared_error,
)

import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.float_format', '{:.4f}'.format)
sns.set_theme(style='whitegrid', palette='muted', font_scale=1.05)

COLORS = {
    'primary'  : '#2E86AB',
    'secondary': '#E84855',
    'accent'   : '#3BB273',
    'warning'  : '#F18F01',
    'purple'   : '#7B2D8B',
    'palette'  : ['#2E86AB','#E84855','#3BB273','#F18F01','#7B2D8B','#F4D35E'],
}
print('✅ Libraries loaded!')

---
## 1️⃣ Dataset Setup

In [ ]:
np.random.seed(42)

X_clf, y_clf = make_classification(
    n_samples=1000, n_features=20, n_informative=10,
    n_redundant=5, n_classes=2, weights=[0.5, 0.5], random_state=42
)
X_clf_df = pd.DataFrame(X_clf, columns=[f'F{i+1:02d}' for i in range(20)])
y_clf_s  = pd.Series(y_clf, name='Target')

X_reg, y_reg = make_regression(
    n_samples=800, n_features=15, n_informative=8, noise=25, random_state=42
)
X_reg_df = pd.DataFrame(X_reg, columns=[f'R{i+1:02d}' for i in range(15)])
y_reg_s  = pd.Series(y_reg, name='Target')

sc_clf = StandardScaler(); sc_reg = StandardScaler()
Xc_tr, Xc_te, yc_tr, yc_te = train_test_split(
    X_clf_df, y_clf_s, test_size=0.2, stratify=y_clf_s, random_state=42)
Xr_tr, Xr_te, yr_tr, yr_te = train_test_split(
    X_reg_df, y_reg_s, test_size=0.2, random_state=42)
Xc_tr_sc = sc_clf.fit_transform(Xc_tr); Xc_te_sc = sc_clf.transform(Xc_te)
Xr_tr_sc = sc_reg.fit_transform(Xr_tr); Xr_te_sc = sc_reg.transform(Xr_te)

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
kf  = KFold(n_splits=5, shuffle=True, random_state=42)

print(f'Classification: {X_clf_df.shape} | classes={dict(y_clf_s.value_counts().sort_index())}')
print(f'  Train={Xc_tr.shape}  Test={Xc_te.shape}')
print(f'Regression    : {X_reg_df.shape}')

---
## 2️⃣ Stacking Intuition — Out-of-Fold Predictions

> **Stacking (Stacked Generalization)** trains a meta-learner on the  
> **out-of-fold (OOF) predictions** of base learners — not on the raw features.
>
> ```
> Level 0 — Base Learners:          Level 1 — Meta-Learner:
>   ┌─────────────────────┐
>   │  Model A (LR)       │──-> OOF preds A ─┐
>   │  Model B (RF)       │──-> OOF preds B ─┼──-> Meta-Learner ──-> Final prediction
>   │  Model C (SVM)      │──-> OOF preds C ─┘
>   │  Model D (KNN)      │──-> OOF preds D ─┘
>   └─────────────────────┘
>
> Why OOF?
>   If base models are trained and evaluated on the SAME fold,
>   meta-learner sees overfitted predictions -> data leakage!
>   OOF predictions are unbiased -> meta-learner learns true errors.
> ```


In [ ]:
# Visualize OOF concept
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# OOF diagram
n_folds = 5
n_samples = 20
fold_size = n_samples // n_folds

ax = axes[0]
for fold in range(n_folds):
    test_start = fold * fold_size
    test_end   = test_start + fold_size
    # Train blocks
    for s in range(n_samples):
        if test_start <= s < test_end:
            color = COLORS['secondary']   # test (OOF)
            label = 'OOF (test)' if s == test_start else ''
        else:
            color = COLORS['primary']     # train
            label = 'Train' if s == 0 and fold == 0 else ''
        ax.barh(fold, 1, left=s, color=color, edgecolor='white',
                height=0.6, label=label)

ax.set_xlabel('Sample Index', fontsize=11)
ax.set_ylabel('Fold', fontsize=11)
ax.set_yticks(range(n_folds))
ax.set_yticklabels([f'Fold {i+1}' for i in range(n_folds)])
ax.set_title('Out-of-Fold Prediction Strategy
(OOF covers entire training set)',
             fontsize=12, fontweight='bold')
from matplotlib.patches import Patch
ax.legend(handles=[
    Patch(color=COLORS['primary'],   label='Train fold'),
    Patch(color=COLORS['secondary'], label='OOF / Test fold'),
], fontsize=10, loc='lower right')

# Stacking pipeline diagram
ax2 = axes[1]
ax2.axis('off')
pipeline_text = 'STACKING PIPELINE

Step 1: K-Fold OOF Training (Level 0)
  For each fold k=1..K:
    Train Model A,B,C,D on k-1 folds
    Predict on fold k -> OOF predictions

Step 2: Collect OOF Predictions
  Meta-features = [OOF_A | OOF_B | OOF_C | OOF_D]
  Shape: (n_train, n_base_models)

Step 3: Train Meta-Learner (Level 1)
  meta_model.fit(meta_features, y_train)

Step 4: Inference
  Retrain base models on FULL training set
  Predict test set -> feed to meta-learner -> final pred'
ax2.text(0.05, 0.95, pipeline_text, transform=ax2.transAxes,
         fontsize=10, verticalalignment='top', fontfamily='monospace',
         bbox=dict(boxstyle='round', facecolor='#f0f4f8', alpha=0.8))
ax2.set_title('Stacking — Conceptual Pipeline', fontsize=12, fontweight='bold')

plt.suptitle('Stacking — Out-of-Fold Strategy', fontsize=14, fontweight='bold')
plt.tight_layout(); plt.show()

---
## 3️⃣ StackingClassifier — sklearn API

> sklearn's `StackingClassifier` handles OOF training automatically.
>
> - `estimators` — list of (name, model) base learners (Level 0)
> - `final_estimator` — meta-learner (Level 1)
> - `cv` — folds for OOF generation (default: 5)
> - `stack_method` — 'predict_proba', 'decision_function', or 'predict'
> - `passthrough` — whether to include original features in meta-learner input


In [ ]:
# ── Define diverse base learners ─────────────────────────────────────────
base_learners = [
    ('lr',  LogisticRegression(max_iter=1000, C=1.0, random_state=42)),
    ('rf',  RandomForestClassifier(n_estimators=100, random_state=42)),
    ('gbm', GradientBoostingClassifier(n_estimators=100, learning_rate=0.05,
                                        max_depth=3, random_state=42)),
    ('svm', SVC(probability=True, kernel='rbf', C=1.0, random_state=42)),
    ('knn', KNeighborsClassifier(n_neighbors=7)),
    ('gnb', GaussianNB()),
]

# ── Meta-learner: Logistic Regression ─────────────────────────────────────
stack_clf = StackingClassifier(
    estimators=base_learners,
    final_estimator=LogisticRegression(max_iter=1000, C=0.1, random_state=42),
    cv=skf,
    stack_method='predict_proba',
    passthrough=False,
    n_jobs=-1,
)
stack_clf.fit(Xc_tr_sc, yc_tr)

stack_tr_auc = roc_auc_score(yc_tr, stack_clf.predict_proba(Xc_tr_sc)[:,1])
stack_te_auc = roc_auc_score(yc_te, stack_clf.predict_proba(Xc_te_sc)[:,1])
stack_te_acc = accuracy_score(yc_te, stack_clf.predict(Xc_te_sc))

print('StackingClassifier (6 base learners + LR meta):')
print(f'  Train AUC : {stack_tr_auc:.4f}')
print(f'  Test AUC  : {stack_te_auc:.4f}')
print(f'  Test Acc  : {stack_te_acc:.4f}')
print(f'  Overfit   : {stack_tr_auc - stack_te_auc:.4f}')

# Compare base learners individually vs stacked
print('
Individual Base Learner Performance:')
ind_rows = []
for name, model in base_learners:
    model.fit(Xc_tr_sc, yc_tr)
    tr = roc_auc_score(yc_tr, model.predict_proba(Xc_tr_sc)[:,1])
    te = roc_auc_score(yc_te, model.predict_proba(Xc_te_sc)[:,1])
    ind_rows.append({'Model': name.upper(), 'Train AUC': round(tr,4),
                     'Test AUC': round(te,4), 'Type': 'Base'})
    print(f'  {name.upper():4s}: Train={tr:.4f} | Test={te:.4f}')

ind_rows.append({'Model': 'STACK', 'Train AUC': round(stack_tr_auc,4),
                 'Test AUC': round(stack_te_auc,4), 'Type': 'Stack'})
ind_df = pd.DataFrame(ind_rows)

fig, ax = plt.subplots(figsize=(13, 5))
colors_bar = [COLORS['primary'] if t=='Base' else COLORS['secondary']
              for t in ind_df['Type']]
bars = ax.bar(ind_df['Model'], ind_df['Test AUC'],
              color=colors_bar, alpha=0.85, edgecolor='white')
ax.axhline(ind_df[ind_df['Type']=='Base']['Test AUC'].max(),
           color=COLORS['warning'], linestyle='--', linewidth=2,
           label=f'Best base model')
for bar, val in zip(bars, ind_df['Test AUC']):
    ax.text(bar.get_x()+bar.get_width()/2, val+0.003,
            f'{val:.4f}', ha='center', fontsize=10)
from matplotlib.patches import Patch
ax.legend(handles=[
    Patch(color=COLORS['primary'],   label='Base Learner'),
    Patch(color=COLORS['secondary'], label='Stacked Ensemble'),
] + ax.get_legend_handles_labels()[0], fontsize=10)
ax.set_ylabel('Test ROC-AUC', fontsize=11)
ax.set_title('StackingClassifier vs Individual Base Learners',
             fontsize=13, fontweight='bold')
ax.set_ylim([0.75, 1.0])
plt.tight_layout(); plt.show()

---
## 4️⃣ Meta-Learner Comparison

> The meta-learner learns **how to best combine base model predictions**.  
> Simple, regularized models (Logistic Regression, Ridge) work best —  
> they prevent the meta-learner from overfitting the OOF predictions.


In [ ]:
meta_learners = {
    'LR (C=1.0)'   : LogisticRegression(max_iter=1000, C=1.0, random_state=42),
    'LR (C=0.1)'   : LogisticRegression(max_iter=1000, C=0.1, random_state=42),
    'LR (C=0.01)'  : LogisticRegression(max_iter=1000, C=0.01, random_state=42),
    'Random Forest': RandomForestClassifier(n_estimators=100, random_state=42),
    'GBM'          : GradientBoostingClassifier(n_estimators=100, random_state=42),
    'Naive Bayes'  : GaussianNB(),
}

meta_results = []
print('Meta-Learner Comparison:')
for name, meta in meta_learners.items():
    stack = StackingClassifier(
        estimators=base_learners,
        final_estimator=meta,
        cv=skf,
        stack_method='predict_proba',
        passthrough=False,
        n_jobs=-1,
    )
    stack.fit(Xc_tr_sc, yc_tr)
    tr_auc = roc_auc_score(yc_tr, stack.predict_proba(Xc_tr_sc)[:,1])
    te_auc = roc_auc_score(yc_te, stack.predict_proba(Xc_te_sc)[:,1])
    meta_results.append({
        'Meta-Learner': name,
        'Train AUC'   : round(tr_auc, 4),
        'Test AUC'    : round(te_auc, 4),
        'Overfit'     : round(tr_auc - te_auc, 4),
    })
    print(f'  {name:20s}: Train={tr_auc:.4f} | Test={te_auc:.4f} | '
          f'Gap={tr_auc-te_auc:.4f}')

meta_df = pd.DataFrame(meta_results).sort_values('Test AUC', ascending=False)

fig, axes = plt.subplots(1, 2, figsize=(16, 5))
x = np.arange(len(meta_df)); w = 0.35
axes[0].bar(x-w/2, meta_df['Train AUC'], w, label='Train AUC',
            color=COLORS['primary'], alpha=0.85)
axes[0].bar(x+w/2, meta_df['Test AUC'],  w, label='Test AUC',
            color=COLORS['accent'], alpha=0.85)
axes[0].set_xticks(x)
axes[0].set_xticklabels(meta_df['Meta-Learner'], rotation=25, ha='right', fontsize=9)
axes[0].set_ylabel('ROC-AUC', fontsize=11)
axes[0].set_title('Meta-Learner Comparison — Train vs Test AUC',
                  fontsize=12, fontweight='bold')
axes[0].legend(fontsize=10); axes[0].set_ylim([0.75, 1.05])

gap_colors = [COLORS['secondary'] if g > 0.05 else COLORS['accent']
              for g in meta_df['Overfit']]
axes[1].bar(meta_df['Meta-Learner'], meta_df['Overfit'],
            color=gap_colors, alpha=0.85, edgecolor='white')
axes[1].set_xticklabels(meta_df['Meta-Learner'], rotation=25, ha='right', fontsize=9)
axes[1].axhline(0.05, color=COLORS['warning'], linestyle='--',
                linewidth=2, label='Threshold (0.05)')
axes[1].set_ylabel('Train − Test AUC', fontsize=11)
axes[1].set_title('Overfitting Gap per Meta-Learner', fontsize=12, fontweight='bold')
axes[1].legend(fontsize=10)

plt.suptitle('Meta-Learner Comparison', fontsize=14, fontweight='bold')
plt.tight_layout(); plt.show()

---
## 5️⃣ Manual Stacking — Step-by-Step OOF Implementation

> Building stacking manually gives full control and transparency  
> over the OOF generation process.


In [ ]:
def manual_stacking(X_train, y_train, X_test, base_models,
                     meta_model, cv_folds=5):
    """
    Manual stacking with OOF predictions.

    Steps:
      1. For each base model:
         - Generate OOF predictions via K-Fold CV on training set
         - Retrain on full training set -> predict test set
      2. Stack OOF predictions as meta-features
      3. Train meta-learner on meta-features (OOF)
      4. Final prediction: meta-learner on test meta-features
    """
    skf_m = StratifiedKFold(n_splits=cv_folds, shuffle=True, random_state=42)
    n_tr  = X_train.shape[0]
    n_te  = X_test.shape[0]
    n_m   = len(base_models)

    oof_preds  = np.zeros((n_tr, n_m))  # OOF for meta-learner training
    test_preds = np.zeros((n_te, n_m))  # Test predictions for final inference

    for m_idx, (name, model) in enumerate(base_models):
        print(f'  Processing {name}...')
        test_fold_preds = np.zeros((n_te, cv_folds))

        for fold, (tr_idx, val_idx) in enumerate(skf_m.split(X_train, y_train)):
            X_tr_f  = X_train[tr_idx];  y_tr_f = y_train[tr_idx]
            X_val_f = X_train[val_idx]

            model_clone = model.__class__(**model.get_params())
            model_clone.fit(X_tr_f, y_tr_f)

            oof_preds[val_idx, m_idx] = model_clone.predict_proba(X_val_f)[:,1]
            test_fold_preds[:, fold]   = model_clone.predict_proba(X_test)[:,1]

        # Retrain on full training set
        model.fit(X_train, y_train)
        test_preds[:, m_idx] = test_fold_preds.mean(axis=1)

    # Train meta-learner on OOF predictions
    meta_model.fit(oof_preds, y_train)

    # Final predictions
    final_preds = meta_model.predict_proba(test_preds)[:,1]

    oof_auc  = roc_auc_score(y_train, meta_model.predict_proba(oof_preds)[:,1])
    test_auc = roc_auc_score(y_test_manual, final_preds)

    return {
        'oof_preds'   : oof_preds,
        'test_preds'  : test_preds,
        'final_preds' : final_preds,
        'meta_model'  : meta_model,
        'oof_auc'     : oof_auc,
        'test_auc'    : test_auc,
    }

# Global for use in function
y_test_manual = yc_te.values

manual_base = [
    ('LR',  LogisticRegression(max_iter=1000, random_state=42)),
    ('RF',  RandomForestClassifier(n_estimators=100, random_state=42)),
    ('GBM', GradientBoostingClassifier(n_estimators=100, learning_rate=0.05,
                                        max_depth=3, random_state=42)),
    ('KNN', KNeighborsClassifier(n_neighbors=7)),
]

print('Manual Stacking (5-fold OOF):')
manual_result = manual_stacking(
    Xc_tr_sc, yc_tr.values, Xc_te_sc,
    base_models=manual_base,
    meta_model=LogisticRegression(max_iter=1000, C=0.1, random_state=42),
    cv_folds=5
)
print(f'\n  OOF AUC  : {manual_result["oof_auc"]:.4f}')
print(f'  Test AUC : {manual_result["test_auc"]:.4f}')

# Show OOF meta-features
oof_df = pd.DataFrame(manual_result['oof_preds'],
                       columns=['OOF_LR','OOF_RF','OOF_GBM','OOF_KNN'])
print(f'\nOOF meta-features shape: {oof_df.shape}')
print(oof_df.head().round(4).to_string())

---
## 6️⃣ Base Learner Diversity — Why Diverse Models Stack Better

> The key to effective stacking is **diverse base learners** that make  
> different types of errors. Low pairwise correlation between OOF predictions  
> -> more complementary information -> better meta-learner.
>
> **Diversity sources:**
> - Different algorithms (tree, linear, kernel, distance-based)
> - Different hyperparameters
> - Different feature subsets
> - Different training data subsets


In [ ]:
# Generate OOF predictions for all base learners
diverse_base = [
    ('LogReg'  , LogisticRegression(max_iter=1000, random_state=42)),
    ('RF'      , RandomForestClassifier(n_estimators=100, random_state=42)),
    ('GBM'     , GradientBoostingClassifier(n_estimators=100, learning_rate=0.05,
                                              max_depth=3, random_state=42)),
    ('SVM'     , SVC(probability=True, kernel='rbf', random_state=42)),
    ('KNN'     , KNeighborsClassifier(n_neighbors=7)),
    ('NaiveBayes', GaussianNB()),
]

# Collect OOF predictions via cross_val_predict
oof_dict = {}
for name, model in diverse_base:
    oof = cross_val_predict(model, Xc_tr_sc, yc_tr, cv=skf,
                             method='predict_proba', n_jobs=-1)[:,1]
    oof_dict[name] = oof

oof_meta_df = pd.DataFrame(oof_dict)

# Pairwise correlation
corr_matrix = oof_meta_df.corr()
print('OOF Prediction Pairwise Correlation:')
print(corr_matrix.round(3).to_string())
print(f'
Mean pairwise correlation: {corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool)).stack().mean():.3f}')

fig, axes = plt.subplots(1, 2, figsize=(17, 6))

sns.heatmap(corr_matrix, annot=True, fmt='.3f', cmap='RdYlGn_r',
            ax=axes[0], linewidths=0.5, linecolor='white',
            vmin=0.5, vmax=1.0, cbar_kws={'shrink':0.8})
axes[0].set_title('OOF Prediction Correlation
(lower = more diverse = better stacking)',
                  fontsize=12, fontweight='bold')

# Scatter: most vs least correlated pair
cols    = list(oof_dict.keys())
corr_vals = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool)
                               ).stack()
most_corr = corr_vals.idxmax()
least_corr = corr_vals.idxmin()

axes[1].scatter(oof_meta_df[most_corr[0]], oof_meta_df[most_corr[1]],
                alpha=0.3, s=20, color=COLORS['secondary'],
                label=f'Most corr: {most_corr[0]} vs {most_corr[1]} '
                      f'(r={corr_vals[most_corr]:.3f})')
axes[1].scatter(oof_meta_df[least_corr[0]], oof_meta_df[least_corr[1]],
                alpha=0.3, s=20, color=COLORS['accent'],
                label=f'Least corr: {least_corr[0]} vs {least_corr[1]} '
                      f'(r={corr_vals[least_corr]:.3f})')
axes[1].set_xlabel('OOF Prediction (Model A)', fontsize=11)
axes[1].set_ylabel('OOF Prediction (Model B)', fontsize=11)
axes[1].set_title('Most vs Least Correlated OOF Predictions',
                  fontsize=12, fontweight='bold')
axes[1].legend(fontsize=9)

plt.suptitle('Base Learner Diversity Analysis', fontsize=14, fontweight='bold')
plt.tight_layout(); plt.show()

---
## 7️⃣ Passthrough Features — Include Original Features in Meta-Learner

> `passthrough=True` passes the **original features** along with OOF predictions  
> to the meta-learner — giving it more context to make its final decision.
>
> Useful when base models don't capture all signal in the features.


In [ ]:
results_pt = []
for pt, label in [(False, 'No passthrough'), (True, 'With passthrough')]:
    stack = StackingClassifier(
        estimators=base_learners,
        final_estimator=LogisticRegression(max_iter=1000, C=0.1, random_state=42),
        cv=skf, stack_method='predict_proba',
        passthrough=pt, n_jobs=-1,
    )
    stack.fit(Xc_tr_sc, yc_tr)
    tr_auc = roc_auc_score(yc_tr, stack.predict_proba(Xc_tr_sc)[:,1])
    te_auc = roc_auc_score(yc_te, stack.predict_proba(Xc_te_sc)[:,1])
    n_meta_feats = (len(base_learners) * 2 +
                    (Xc_tr_sc.shape[1] if pt else 0))
    results_pt.append({
        'Config'        : label,
        'Meta-features' : n_meta_feats,
        'Train AUC'     : round(tr_auc, 4),
        'Test AUC'      : round(te_auc, 4),
        'Overfit'       : round(tr_auc - te_auc, 4),
    })
    print(f'  {label}: meta_feats={n_meta_feats} | '
          f'Train={tr_auc:.4f} | Test={te_auc:.4f}')

pt_df = pd.DataFrame(results_pt)
print(pt_df.to_string(index=False))

fig, ax = plt.subplots(figsize=(9, 4))
x = np.arange(2); w = 0.35
ax.bar(x-w/2, pt_df['Train AUC'], w, label='Train AUC',
       color=COLORS['primary'], alpha=0.85)
ax.bar(x+w/2, pt_df['Test AUC'],  w, label='Test AUC',
       color=COLORS['accent'], alpha=0.85)
for i, (tr, te) in enumerate(zip(pt_df['Train AUC'], pt_df['Test AUC'])):
    ax.text(i-w/2, tr+0.003, f'{tr:.4f}', ha='center', fontsize=11)
    ax.text(i+w/2, te+0.003, f'{te:.4f}', ha='center', fontsize=11)
ax.set_xticks(x); ax.set_xticklabels(pt_df['Config'], fontsize=11)
ax.set_ylabel('ROC-AUC', fontsize=11)
ax.set_title('Passthrough Features — Effect on Stacking Performance',
             fontsize=13, fontweight='bold')
ax.legend(fontsize=10); ax.set_ylim([0.8, 1.02])
plt.tight_layout(); plt.show()

---
## 8️⃣ Multi-Level Stacking — Level 0 -> Level 1 -> Level 2

> Multi-level stacking adds another layer on top of Level 1 predictions.  
> In practice, gains beyond 2 levels are minimal and risk overfitting.
>
> ```
> Level 0: LR, RF, GBM, SVM, KNN -> OOF predictions
> Level 1: Stack on Level 0 OOFs -> LR, RF meta-predictions
> Level 2: Final meta-learner on Level 1 outputs
> ```


In [ ]:
# Level 0 base learners
level0 = [
    ('lr',  LogisticRegression(max_iter=1000, C=1.0, random_state=42)),
    ('rf',  RandomForestClassifier(n_estimators=100, random_state=42)),
    ('gbm', GradientBoostingClassifier(n_estimators=100, learning_rate=0.05,
                                        max_depth=3, random_state=42)),
    ('knn', KNeighborsClassifier(n_neighbors=7)),
]

# Level 1: stacking on Level 0 OOFs, using two different meta-learners
level1_lr  = StackingClassifier(
    estimators=level0,
    final_estimator=LogisticRegression(max_iter=1000, C=0.1, random_state=42),
    cv=skf, stack_method='predict_proba', passthrough=False, n_jobs=-1,
)
level1_rf  = StackingClassifier(
    estimators=level0,
    final_estimator=RandomForestClassifier(n_estimators=50, random_state=42),
    cv=skf, stack_method='predict_proba', passthrough=False, n_jobs=-1,
)

# Level 2: stack on Level 1 meta-predictions
level2 = StackingClassifier(
    estimators=[
        ('l1_lr', level1_lr),
        ('l1_rf', level1_rf),
    ],
    final_estimator=LogisticRegression(max_iter=1000, C=0.01, random_state=42),
    cv=skf, stack_method='predict_proba', passthrough=False, n_jobs=-1,
)
level2.fit(Xc_tr_sc, yc_tr)

l2_tr_auc = roc_auc_score(yc_tr, level2.predict_proba(Xc_tr_sc)[:,1])
l2_te_auc = roc_auc_score(yc_te, level2.predict_proba(Xc_te_sc)[:,1])

print('Multi-Level Stacking Results:')

# Compare 1-level vs 2-level
level1_lr_fit = StackingClassifier(
    estimators=level0,
    final_estimator=LogisticRegression(max_iter=1000, C=0.1, random_state=42),
    cv=skf, stack_method='predict_proba', n_jobs=-1,
)
level1_lr_fit.fit(Xc_tr_sc, yc_tr)
l1_tr_auc = roc_auc_score(yc_tr, level1_lr_fit.predict_proba(Xc_tr_sc)[:,1])
l1_te_auc = roc_auc_score(yc_te, level1_lr_fit.predict_proba(Xc_te_sc)[:,1])

level_df = pd.DataFrame([
    {'Level': 'Level 1 (4 base + LR meta)',
     'Train AUC': round(l1_tr_auc,4), 'Test AUC': round(l1_te_auc,4),
     'Gap': round(l1_tr_auc-l1_te_auc,4)},
    {'Level': 'Level 2 (L0->L1->L2)',
     'Train AUC': round(l2_tr_auc,4), 'Test AUC': round(l2_te_auc,4),
     'Gap': round(l2_tr_auc-l2_te_auc,4)},
])
print(level_df.to_string(index=False))
print('
Note: Level 2 adds complexity — may not always improve over Level 1.')

---
## 9️⃣ StackingRegressor — Regression Stacking

> `StackingRegressor` works identically to the classifier version —  
> base learners predict continuous values, meta-learner combines them.


In [ ]:
from sklearn.linear_model import ElasticNet
from sklearn.ensemble import ExtraTreesRegressor

reg_base = [
    ('ridge', Ridge(alpha=1.0)),
    ('rf',    RandomForestRegressor(n_estimators=100, random_state=42)),
    ('gbm',   GradientBoostingRegressor(n_estimators=100, learning_rate=0.05,
                                         max_depth=3, random_state=42)),
    ('knn',   KNeighborsRegressor(n_neighbors=7)),
    ('en',    ElasticNet(alpha=0.1, l1_ratio=0.5, max_iter=5000)),
]

stack_reg = StackingRegressor(
    estimators=reg_base,
    final_estimator=RidgeCV(alphas=[0.01, 0.1, 1.0, 10.0]),
    cv=kf,
    passthrough=False,
    n_jobs=-1,
)
stack_reg.fit(Xr_tr_sc, yr_tr)

y_pred_stack = stack_reg.predict(Xr_te_sc)
r2_stack  = r2_score(yr_te, y_pred_stack)
rmse_stack = np.sqrt(mean_squared_error(yr_te, y_pred_stack))

print('StackingRegressor (5 base + Ridge meta):')
print(f'  Test R²   : {r2_stack:.4f}')
print(f'  Test RMSE : {rmse_stack:.4f}')

# Compare individual regressors vs stacked
print('
Individual Regressor Performance:')
reg_rows = []
for name, model in reg_base:
    model.fit(Xr_tr_sc, yr_tr)
    r2 = r2_score(yr_te, model.predict(Xr_te_sc))
    reg_rows.append({'Model': name.upper(), 'R²': round(r2,4), 'Type': 'Base'})
    print(f'  {name.upper():6s}: R²={r2:.4f}')
reg_rows.append({'Model':'STACK','R²': round(r2_stack,4), 'Type':'Stack'})
reg_df = pd.DataFrame(reg_rows).sort_values('R²', ascending=False)

fig, ax = plt.subplots(figsize=(11, 5))
bar_colors = [COLORS['primary'] if t=='Base' else COLORS['secondary']
              for t in reg_df['Type']]
bars = ax.bar(reg_df['Model'], reg_df['R²'], color=bar_colors,
              alpha=0.85, edgecolor='white')
for bar, val in zip(bars, reg_df['R²']):
    ax.text(bar.get_x()+bar.get_width()/2, val+0.003,
            f'{val:.4f}', ha='center', fontsize=10)
ax.set_ylabel('Test R²', fontsize=11)
ax.set_title('StackingRegressor vs Individual Regressors',
             fontsize=13, fontweight='bold')
from matplotlib.patches import Patch
ax.legend(handles=[
    Patch(color=COLORS['primary'],   label='Base Regressor'),
    Patch(color=COLORS['secondary'], label='Stacked Ensemble'),
], fontsize=10)
ax.set_ylim([0.5, 1.05])
plt.tight_layout(); plt.show()

---
## 🔟 Voting vs Stacking — Learned vs Fixed Combination

> **VotingClassifier** combines predictions using a **fixed rule**:
> - Hard voting: majority class vote
> - Soft voting: average of predicted probabilities
>
> **StackingClassifier** uses a **learned combination** — the meta-learner  
> learns the optimal weights to assign to each base model.


In [ ]:
# Voting (soft)
voting_soft = VotingClassifier(
    estimators=base_learners, voting='soft', n_jobs=-1
)
voting_soft.fit(Xc_tr_sc, yc_tr)
v_tr = roc_auc_score(yc_tr, voting_soft.predict_proba(Xc_tr_sc)[:,1])
v_te = roc_auc_score(yc_te, voting_soft.predict_proba(Xc_te_sc)[:,1])

# Stacking (best from earlier)
s_tr = stack_tr_auc
s_te = stack_te_auc

# Best individual base model
best_base_te = max([roc_auc_score(yc_te, m.predict_proba(Xc_te_sc)[:,1])
                    for _, m in base_learners])

comparison = pd.DataFrame([
    {'Method': 'Best Base Model',      'Test AUC': round(best_base_te,4), 'Type': 'Single'},
    {'Method': 'Soft Voting',           'Test AUC': round(v_te,4),        'Type': 'Voting'},
    {'Method': 'Stacking (LR meta)',    'Test AUC': round(s_te,4),        'Type': 'Stacking'},
]).sort_values('Test AUC', ascending=False)

print('Voting vs Stacking vs Best Individual:')
print(comparison.to_string(index=False))

print(f'
Soft Voting  : Train={v_tr:.4f} | Test={v_te:.4f} | Gap={v_tr-v_te:.4f}')
print(f'Stacking     : Train={s_tr:.4f} | Test={s_te:.4f} | Gap={s_tr-s_te:.4f}')

fig, ax = plt.subplots(figsize=(9, 4))
type_colors = {'Single':COLORS['warning'],'Voting':COLORS['primary'],
               'Stacking':COLORS['secondary']}
bar_colors = [type_colors[t] for t in comparison['Type']]
bars = ax.bar(comparison['Method'], comparison['Test AUC'],
              color=bar_colors, alpha=0.85, edgecolor='white')
for bar, val in zip(bars, comparison['Test AUC']):
    ax.text(bar.get_x()+bar.get_width()/2, val+0.002,
            f'{val:.4f}', ha='center', fontsize=11)
ax.set_ylabel('Test ROC-AUC', fontsize=11)
ax.set_title('Voting vs Stacking vs Best Single Model',
             fontsize=13, fontweight='bold')
ax.set_ylim([0.85, 1.0])
plt.tight_layout(); plt.show()

---
## 1️⃣1️⃣ Full Leaderboard — All Models and Ensembles


In [ ]:
all_models = {
    'LR (single)'        : LogisticRegression(max_iter=1000, random_state=42),
    'RF (single)'        : RandomForestClassifier(n_estimators=100, random_state=42),
    'GBM (single)'       : GradientBoostingClassifier(n_estimators=100,
                                                       learning_rate=0.05,
                                                       max_depth=3, random_state=42),
    'SVM (single)'       : SVC(probability=True, random_state=42),
    'KNN (single)'       : KNeighborsClassifier(n_neighbors=7),
    'Bagging (DT)'       : BaggingClassifier(
        DecisionTreeClassifier(max_depth=None),
        n_estimators=100, random_state=42, n_jobs=-1),
    'Soft Voting'        : VotingClassifier(estimators=base_learners,
                                            voting='soft', n_jobs=-1),
    'Stacking (LR meta)' : StackingClassifier(
        estimators=base_learners,
        final_estimator=LogisticRegression(max_iter=1000, C=0.1, random_state=42),
        cv=skf, stack_method='predict_proba', n_jobs=-1),
    'Stacking (RF meta)' : StackingClassifier(
        estimators=base_learners,
        final_estimator=RandomForestClassifier(n_estimators=50, random_state=42),
        cv=skf, stack_method='predict_proba', n_jobs=-1),
}

lb_rows = []
for name, model in all_models.items():
    model.fit(Xc_tr_sc, yc_tr)
    tr_auc = roc_auc_score(yc_tr, model.predict_proba(Xc_tr_sc)[:,1])
    te_auc = roc_auc_score(yc_te, model.predict_proba(Xc_te_sc)[:,1])
    te_acc = accuracy_score(yc_te, model.predict(Xc_te_sc))
    lb_rows.append({'Model': name, 'Train AUC': round(tr_auc,4),
                    'Test AUC': round(te_auc,4), 'Test Acc': round(te_acc,4),
                    'Overfit': round(tr_auc-te_auc,4)})
    print(f'  {name:25s}: Train={tr_auc:.4f} | Test={te_auc:.4f}')

lb_df = pd.DataFrame(lb_rows).sort_values('Test AUC', ascending=False)
print('
📊 Full Leaderboard:')
print(lb_df.to_string(index=False))

fig, ax = plt.subplots(figsize=(14, 6))
bars = ax.barh(lb_df['Model'], lb_df['Test AUC'],
               color=[COLORS['secondary'] if 'Stack' in m else
                      COLORS['primary'] if 'single' in m else
                      COLORS['accent'] for m in lb_df['Model']],
               alpha=0.85, edgecolor='white')
ax.axvline(lb_df[lb_df['Model'].str.contains('single')]['Test AUC'].max(),
           color=COLORS['warning'], linestyle='--', linewidth=2,
           label='Best single model')
for bar, val in zip(bars, lb_df['Test AUC']):
    ax.text(val+0.002, bar.get_y()+bar.get_height()/2,
            f'{val:.4f}', va='center', fontsize=10)
from matplotlib.patches import Patch
ax.legend(handles=[
    Patch(color=COLORS['secondary'], label='Stacking'),
    Patch(color=COLORS['accent'],    label='Other Ensemble'),
    Patch(color=COLORS['primary'],   label='Single Model'),
] + ax.get_legend_handles_labels()[0], fontsize=9)
ax.set_xlabel('Test ROC-AUC', fontsize=11)
ax.set_title('Full Leaderboard — Single Models vs Ensembles',
             fontsize=13, fontweight='bold')
ax.set_xlim([0.8, 1.02])
plt.tight_layout(); plt.show()

---
## ✅ 12. Summary & Golden Rules

| Method | Combination | Learns Weights | Base Learner Type |
|--------|:-----------:|:--------------:|:-----------------:|
| **Voting (soft)** | Average probs | ❌ Fixed equal | Any |
| **Voting (hard)** | Majority vote | ❌ Fixed equal | Any |
| **Bagging** | Average | ❌ Fixed equal | Same model |
| **Stacking** | Meta-learner | ✅ Learned | Diverse models |

### 🔑 Golden Rules

1. **Diversity is the key** — diverse base learners (low OOF correlation) stack better
2. **Always use OOF predictions** — never train meta-learner on in-fold predictions (leakage!)
3. **Simple meta-learners work best** — Logistic Regression with regularization (C=0.1)
4. **2 levels is almost always enough** — beyond that, gains are marginal
5. **passthrough=True helps when features carry signal base models miss**
6. **More base learners = more diverse = better** — but diminishing returns past 5–7
7. **Include at least one linear model** in base learners for stacking
8. **Stacking is most useful in competitions** — in production, simpler ensembles suffice
9. **OOF AUC ≈ test AUC** — a big gap indicates overfitting in the meta-learner
10. **Cross-validate the full stacking pipeline** — not just the meta-learner

---

## 🔗 Next Steps

- ➡️ `08_Ensemble_Learning/bagging.ipynb` — Parallel variance-reducing ensemble
- ➡️ `08_Ensemble_Learning/boosting.ipynb` — Sequential bias-reducing ensemble
- ➡️ `07_Hyperparameter_Tuning/gridsearchcv.ipynb` — Tune base learners before stacking
- ➡️ `05_Model_Evaluation/cross_validation.ipynb` — Proper CV for stacking evaluation
